# 02 — Silver Layer
Cleans and enriches Bronze data. Applies 8 data quality rules removing 2.7M invalid records.
Broadcast joins with 265-row zone lookup to add borough and zone names.
Partitioned by pickup_year and pickup_month for downstream query performance.

In [0]:
# ADLS auth 
client_id     = dbutils.secrets.get(scope="kv-scope", key="sp-client-id")
tenant_id     = dbutils.secrets.get(scope="kv-scope", key="sp-tenant-id")
client_secret = dbutils.secrets.get(scope="kv-scope", key="sp-client-secret")

storage_account = "azurelabadls225"

spark.conf.set(f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net", "OAuth")
spark.conf.set(f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set(f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net", client_id)
spark.conf.set(f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net", f"https://login.microsoftonline.com/{tenant_id}/oauth2/token")
spark.conf.set(f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net", client_secret)

RAW_PATH       = f"abfss://raw@{storage_account}.dfs.core.windows.net"
PROCESSED_PATH = f"abfss://processed@{storage_account}.dfs.core.windows.net"
CURATED_PATH   = f"abfss://curated@{storage_account}.dfs.core.windows.net"

print("Setup complete")

In [0]:
df_bronze = spark.read.format("delta").load(f"{RAW_PATH}/delta/bronze_yellow_taxi")

print(f"Bronze rows: {df_bronze.count():,}")
df_bronze.printSchema()

In [0]:
from pyspark.sql.functions import col, year, unix_timestamp

df_cleaned = (df_bronze
    # Remove negative or zero fares
    .filter(col("fare_amount") > 0)
    # Remove trips with no distance
    .filter(col("trip_distance") > 0)
    # Remove invalid passenger counts
    .filter(col("passenger_count") > 0)
    .filter(col("passenger_count") <= 6)
    # Remove null locations
    .filter(col("PULocationID").isNotNull())
    .filter(col("DOLocationID").isNotNull())
    # Remove null timestamps
    .filter(col("tpep_pickup_datetime").isNotNull())
    .filter(col("tpep_dropoff_datetime").isNotNull())
    # Remove trips where dropoff is before pickup
    .filter(col("tpep_dropoff_datetime") > col("tpep_pickup_datetime"))
    # Only keep 2023 data — removes wrong year garbage rows
    .filter(year(col("tpep_pickup_datetime")) == 2023)
    # Max trip duration 3 hours — removes impossibly long trips
    .filter(
        ((unix_timestamp("tpep_dropoff_datetime") -
          unix_timestamp("tpep_pickup_datetime")) / 60) <= 180
    )
    # Remove duplicates
    .dropDuplicates(["tpep_pickup_datetime", "tpep_dropoff_datetime",
                     "PULocationID", "DOLocationID", "fare_amount"])
)

print(f"Rows after cleaning: {df_cleaned.count():,}")
print(f"Rows removed: {df_bronze.count() - df_cleaned.count():,}")

In [0]:
from pyspark.sql.functions import broadcast

# Read zone lookup — small table (265 rows) perfect for broadcast join
df_zones = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(f"{RAW_PATH}/lookup/taxi_zone_lookup.csv")
)

print(f"Zone lookup rows: {df_zones.count()}")
df_zones.show(3)

# Broadcast join — send the small zones table to ALL executors
# This avoids a shuffle of the 38M row table
df_silver = (df_cleaned
    .join(broadcast(df_zones.alias("pickup_zone")),
          df_cleaned.PULocationID == col("pickup_zone.LocationID"),
          "left")
    .withColumnRenamed("Borough", "pickup_borough")
    .withColumnRenamed("Zone", "pickup_zone_name")
    .withColumnRenamed("service_zone", "pickup_service_zone")
    .drop("LocationID")
    .join(broadcast(df_zones.alias("dropoff_zone")),
          df_cleaned.DOLocationID == col("dropoff_zone.LocationID"),
          "left")
    .withColumnRenamed("Borough", "dropoff_borough")
    .withColumnRenamed("Zone", "dropoff_zone_name")
    .withColumnRenamed("service_zone", "dropoff_service_zone")
    .drop("LocationID")
)

print(f"Broadcast join complete")
print(f"Silver columns: {len(df_silver.columns)}")

In [0]:
from pyspark.sql.functions import (unix_timestamp, round, hour, 
                                    dayofweek, month, year, lit)

df_silver_enriched = (df_silver
    # Trip duration in minutes
    .withColumn("trip_duration_mins",
        round((unix_timestamp("tpep_dropoff_datetime") - 
               unix_timestamp("tpep_pickup_datetime")) / 60, 2))
    # Time features
    .withColumn("pickup_hour",    hour("tpep_pickup_datetime"))
    .withColumn("pickup_day",     dayofweek("tpep_pickup_datetime"))
    .withColumn("pickup_month",   month("tpep_pickup_datetime"))
    .withColumn("pickup_year",    year("tpep_pickup_datetime"))
    # Silver metadata
    .withColumn("silver_pipeline", lit("nyc_taxi_silver"))
)

print(f"Derived columns added")
print(f"Final silver columns: {len(df_silver_enriched.columns)}")

In [0]:
silver_path = f"{PROCESSED_PATH}/delta/silver_yellow_taxi"

(df_silver_enriched
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("pickup_year", "pickup_month")
    .save(silver_path)
)

print("Silver layer written!")

In [0]:
df_verify = spark.read.format("delta").load(f"{PROCESSED_PATH}/delta/silver_yellow_taxi")

print(f"Total rows:    {df_verify.count():,}")
print(f"Total columns: {len(df_verify.columns)}")
df_verify.show(3)